In [2]:
from pathlib import Path

DATA_DIR = Path(r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\data")

CLEANED_DIR = Path(r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\cleaned data")
CLEANED_DIR.mkdir(exist_ok=True)

DB_PATH = Path(r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\ecommerce.db")

assert DATA_DIR.exists(), "Data folder not found."

print("Setup complete.")
print(DATA_DIR)

Setup complete.
C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\data


In [6]:
orders_raw = pd.read_csv(
    r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\data\orders.csv",
    dtype={"customer_id": str}
)

products_raw = pd.read_csv(
    r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\data\products.csv"
)

customers_raw = pd.read_csv(
    r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\data\customers.csv",
    dtype={"customer_id": str}
)

order_items_raw = pd.read_csv(
    r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\data\order_items.csv"
)

print("customers:  ", len(customers_raw), "rows")
print("products:   ", len(products_raw), "rows")
print("orders:     ", len(orders_raw), "rows")
print("order_items:", len(order_items_raw), "rows")

customers:   600 rows
products:    150 rows
orders:      2000 rows
order_items: 5000 rows


In [7]:
customers_raw.head()

,customer_id,customer_name,email,registration_date,customer_type
0,1,Allison Hill,garzaanthony@gmail.com,2025-09-20 07:33:32,PREMIUM
1,2,Jonathan Johnson,shaneramirez@gmail.com,2024-01-16 10:13:48,PREMIUM
2,3,Caitlin Henderson,daviscolin@yahoo.com,2025-11-17 05:11:07,REGULAR
3,4,Renee Blair,maria95@hotmail.com,2025-07-06 07:09:23,REGULAR
4,5,Benjamin Stanley,jamesshawn@yahoo.com,2023-12-06 15:24:52,REGULAR


In [8]:
orders_raw.head()

,order_id,customer_id,order_date,status,region_code
0,1,90,2023-10-19 02:28:19,DELIVERED,CENTRAL
1,2,27,2023-05-20 05:13:57,PLACED,CENTRAL
2,3,395,2023-01-20 04:05:29,DELIVERED,NORTH
3,4,573,2024-01-13 02:05:49,SHIPPED,WEST
4,5,276,2024-04-12 18:59:14,PLACED,SOUTH


In [10]:
products_raw.head()

,product_id,product_name,category,subcategory,cost_price
0,1,Those Bedding,Home,Bedding,404.64
1,2,Section Comic,Books,Comics,345.86
2,3,Room Footwear,Clothing,Footwear,234.90
3,4,West Furniture,Home,Furniture,360.59
4,5,reflect bedding,Home,Bedding,188.42


In [11]:
order_items_raw.head()

,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,669,81,2,635.70,90.2
1,2,575,85,5,699.84,90.2
2,3,1601,59,5,709.56,60.9
3,4,511,5,4,316.89,88.1
4,5,304,136,5,493.49,60.3


In [12]:
print("Missing customer_id rows:", (orders_raw["customer_id"].isna() | (orders_raw["customer_id"] == "")).sum())
print("DD-MM-YYYY format rows:", orders_raw["order_date"].astype(str).str.match(r"^\d{2}-\d{2}-\d{4}$").sum())
print("Negative quantity rows:", (order_items_raw["quantity"] < 0).sum())
print("Invalid email rows:", (~customers_raw["email"].astype(str).str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")).sum())

Missing customer_id rows: 107
DD-MM-YYYY format rows: 145
Negative quantity rows: 157
Invalid email rows: 8


In [13]:
#Data Cleaning
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

def clean_orders(df: pd.DataFrame):
    """Fix date formats, handle NULL customer_ids. Returns (cleaned_df, issues_dict)."""
    df = df.copy()
    issues = {}


    raw = df["customer_id"]
    missing_mask = raw.isna() | raw.astype(str).str.strip().isin(["", "nan", "NULL", "None", "<NA>"])
    issues["missing_customer_id"] = int(missing_mask.sum())
    cleaned_ids = raw.where(~missing_mask, other=pd.NA)
    df["customer_id"] = pd.to_numeric(cleaned_ids, errors="coerce").astype("Int64")

    
    def fix_date(value):
        value = str(value).strip()
        parsed = pd.to_datetime(value, format="%Y-%m-%d %H:%M:%S", errors="coerce")
        if pd.isna(parsed):
            parsed2 = pd.to_datetime(value, format="%d-%m-%Y", errors="coerce")
            if not pd.isna(parsed2):
                return parsed2
            return pd.to_datetime(value, errors="coerce")
        return parsed

    had_dash_fmt = df["order_date"].astype(str).str.match(r"^\d{2}-\d{2}-\d{4}$")
    issues["bad_date_format_fixed"] = int(had_dash_fmt.sum())
    df["order_date"] = df["order_date"].apply(fix_date)
    issues["unparseable_dates"] = int(df["order_date"].isna().sum())

    return df, issues


def clean_products(df: pd.DataFrame):
    """Normalize product_name: trim whitespace, title case. Returns (cleaned_df, issues_dict)."""
    df = df.copy()
    before = df["product_name"].copy()
    df["product_name"] = df["product_name"].astype(str).str.strip().str.title()
    changed = int((before.astype(str).str.strip().str.title() != before).sum())
    return df, {"product_names_normalized": changed}


def validate_emails(df: pd.DataFrame):
    """Return list of customer_ids whose email is missing @ or a domain."""
    invalid_mask = ~df["email"].astype(str).str.match(EMAIL_RE)
    return df.loc[invalid_mask, "customer_id"].tolist()


def check_referential_integrity(orders_df: pd.DataFrame, order_items_df: pd.DataFrame):
    """Return order_items rows whose order_id does not exist in orders_df."""
    valid_order_ids = set(orders_df["order_id"])
    bad_mask = ~order_items_df["order_id"].isin(valid_order_ids)
    return order_items_df.loc[bad_mask]


def clean_order_items(df: pd.DataFrame):
    """Clip discount_percent to [0,100]; report negative/zero quantity counts."""
    df = df.copy()
    issues = {}
    out_of_range = df["discount_percent"].gt(100) | df["discount_percent"].lt(0)
    issues["discount_percent_out_of_range"] = int(out_of_range.sum())
    df["discount_percent"] = df["discount_percent"].clip(lower=0, upper=100)
    issues["negative_quantity_rows_returns"] = int((df["quantity"] < 0).sum())
    issues["zero_quantity_rows"] = int((df["quantity"] == 0).sum())
    return df, issues

print("Cleaning functions defined.")

Cleaning functions defined.


In [15]:
orders_clean, order_issues = clean_orders(orders_raw)
products_clean, product_issues = clean_products(products_raw)
order_items_clean, item_issues = clean_order_items(order_items_raw)
invalid_email_ids = validate_emails(customers_raw)
bad_refs = check_referential_integrity(orders_clean, order_items_clean)

report_lines = ["DATA CLEANING REPORT", "=" * 40, ""]
report_lines.append("orders.csv:")
for k, v in order_issues.items():
    report_lines.append(f"  - {k}: {v}")
report_lines.append("products.csv:")
for k, v in product_issues.items():
    report_lines.append(f"  - {k}: {v}")
report_lines.append("order_items.csv:")
for k, v in item_issues.items():
    report_lines.append(f"  - {k}: {v}")
report_lines.append("customers.csv:")
report_lines.append(f"  - invalid_emails: {len(invalid_email_ids)}")
report_lines.append("referential_integrity:")
report_lines.append(f"  - order_items rows referencing non-existent orders: {len(bad_refs)}")

report_text = "\n".join(report_lines)
print(report_text)

with open(r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\cleaned data\cleaning_report.txt", "w") as f:
    f.write(report_text)

DATA CLEANING REPORT

orders.csv:
  - missing_customer_id: 107
  - bad_date_format_fixed: 145
  - unparseable_dates: 0
products.csv:
  - product_names_normalized: 35
order_items.csv:
  - discount_percent_out_of_range: 0
  - negative_quantity_rows_returns: 157
  - zero_quantity_rows: 0
customers.csv:
  - invalid_emails: 8
referential_integrity:
  - order_items rows referencing non-existent orders: 34


In [16]:
# cleaned CSVs
orders_clean.to_csv(CLEANED_DIR / "orders.csv", index=False)
products_clean.to_csv(CLEANED_DIR / "products.csv", index=False)
order_items_clean.to_csv(CLEANED_DIR / "order_items.csv", index=False)
customers_raw.to_csv(CLEANED_DIR / "customers.csv", index=False)  # emails flagged, not dropped

print("Cleaned files written to:", CLEANED_DIR.resolve())
orders_clean.head()

Cleaned files written to: C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\Cleaned data


,order_id,customer_id,order_date,status,region_code
0,1,90,2023-10-19 02:28:19,DELIVERED,CENTRAL
1,2,27,2023-05-20 05:13:57,PLACED,CENTRAL
2,3,395,2023-01-20 04:05:29,DELIVERED,NORTH
3,4,573,2024-01-13 02:05:49,SHIPPED,WEST
4,5,276,2024-04-12 18:59:14,PLACED,SOUTH


In [22]:

SCHEMA = """
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    email TEXT,
    registration_date TEXT,
    customer_type TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    subcategory TEXT,
    cost_price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    status TEXT,
    region_code TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price REAL,
    discount_percent REAL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE INDEX idx_orders_customer ON orders(customer_id);
CREATE INDEX idx_orders_date ON orders(order_date);
CREATE INDEX idx_items_order ON order_items(order_id);
CREATE INDEX idx_items_product ON order_items(product_id);
"""

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.executescript(SCHEMA)

valid_order_ids = set(orders_clean["order_id"])
before = len(order_items_clean)
order_items_valid = order_items_clean[order_items_clean["order_id"].isin(valid_order_ids)]
dropped = before - len(order_items_valid)

customers_raw.to_sql("customers", conn, if_exists="append", index=False)
products_clean.to_sql("products", conn, if_exists="append", index=False)
orders_clean.to_sql("orders", conn, if_exists="append", index=False)
order_items_valid.to_sql("order_items", conn, if_exists="append", index=False)
conn.commit()

print(f"Loaded into {DB_PATH.resolve()}")
print(f"  customers:   {len(customers_raw)}")
print(f"  products:    {len(products_clean)}")
print(f"  orders:      {len(orders_clean)}")
print(f"  order_items: {len(order_items_valid)} (dropped {dropped} rows with broken order_id FK)")

Loaded into C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week8_Mini Project- E-Commerce Order Analytics System\ecommerce.db
  customers:   600
  products:    150
  orders:      2000
  order_items: 4966 (dropped 34 rows with broken order_id FK)


In [21]:
try:
    conn.close()
except:
    pass

In [23]:
#SQL Analysis
#Query 1 — Total revenue per category
q1 = """
SELECT
    p.category,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""
pd.read_sql_query(q1, conn)

,category,total_revenue
0,Books,1149897.08
1,Clothing,845877.48
2,Home,772052.39
3,Electronics,747658.13


In [24]:
#Query 2 — Top 10 customers by total order value
q2 = """
SELECT
    o.customer_id,
    c.customer_name,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_order_value
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.customer_id IS NOT NULL
GROUP BY o.customer_id, c.customer_name
ORDER BY total_order_value DESC
LIMIT 10;
"""
pd.read_sql_query(q2, conn)

,customer_id,customer_name,total_order_value
0,442,David King,28573.66
1,255,Xavier Rowe,25552.75
2,41,Richard Lawson,24108.70
3,387,Jerry Sullivan,20804.82
4,537,Charles Carpenter,19156.47
5,563,Benjamin Luna,18493.02
6,37,Victoria Johnson,18485.21
7,20,Connor West,17747.01
8,152,Evelyn Martinez,17702.00
9,188,Curtis Maynard,17570.28


In [25]:
#Query 3 — Month-wise order count for the last 12 months
q3 = """
SELECT
    strftime('%Y-%m', order_date) AS order_month,
    COUNT(*) AS order_count
FROM orders
WHERE order_date >= date((SELECT MAX(order_date) FROM orders), '-12 months')
GROUP BY order_month
ORDER BY order_month;
"""
pd.read_sql_query(q3, conn)

,order_month,order_count
0,2024-12,2
1,2025-01,61
2,2025-02,61
3,2025-03,51
4,2025-04,42
5,2025-05,46
6,2025-06,63
7,2025-07,60
8,2025-08,51
9,2025-09,41


In [26]:
#Query 4 — Customers who placed orders but never had anything delivered
q4 = """
SELECT DISTINCT o.customer_id, c.customer_name
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.customer_id IS NOT NULL
  AND o.customer_id NOT IN (
      SELECT customer_id FROM orders WHERE status = 'DELIVERED' AND customer_id IS NOT NULL
  );
"""
pd.read_sql_query(q4, conn)

,customer_id,customer_name
0,8,Austin Gentry
1,12,Jesse Mckay
2,16,Michelle Ross
3,17,Kevin Hall
4,23,Kristi Higgins MD
...,...,...
123,579,Julie Sims
124,581,Casey Floyd
125,583,Timothy Gibson
126,585,Margaret Carr


In [27]:
#Query 5 — Products with more returns than purchases
q5 = """
SELECT
    p.product_id,
    p.product_name,
    SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS total_purchased,
    SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS total_returned
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.product_id, p.product_name
HAVING total_returned > total_purchased;
"""
pd.read_sql_query(q5, conn)

,product_id,product_name,total_purchased,total_returned


In [28]:
#Query 6 — Return rate per category
q6 = """
SELECT
    p.category,
    SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS returned_items,
    SUM(ABS(oi.quantity)) AS total_items,
    ROUND(
        1.0 * SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END)
        / NULLIF(SUM(ABS(oi.quantity)), 0), 4
    ) AS return_rate
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY return_rate DESC;
"""
pd.read_sql_query(q6, conn)

,category,returned_items,total_items,return_rate
0,Electronics,137,3291,0.0416
1,Books,144,4831,0.0298
2,Clothing,96,3578,0.0268
3,Home,70,3089,0.0227


In [29]:
#Query 7 — Running total of revenue per region (window function)
q7 = """
WITH daily AS (
    SELECT
        o.region_code,
        date(o.order_date) AS order_date,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS daily_revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY o.region_code, date(o.order_date)
)
SELECT
    region_code,
    order_date,
    ROUND(daily_revenue, 2) AS daily_revenue,
    ROUND(SUM(daily_revenue) OVER (
        PARTITION BY region_code ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS running_total
FROM daily
ORDER BY region_code, order_date;
"""
pd.read_sql_query(q7, conn)

,region_code,order_date,daily_revenue,running_total
0,CENTRAL,2023-01-05,2015.80,2015.80
1,CENTRAL,2023-01-09,2730.93,4746.73
2,CENTRAL,2023-01-12,2646.52,7393.25
3,CENTRAL,2023-01-19,2274.07,9667.32
4,CENTRAL,2023-01-20,142.05,9809.37
...,...,...,...,...
1542,WEST,2025-12-19,1118.38,710191.50
1543,WEST,2025-12-21,1871.55,712063.05
1544,WEST,2025-12-24,3742.36,715805.42
1545,WEST,2025-12-25,1549.06,717354.48


In [30]:
#Query 8 — Rank products by revenue within category (DENSE_RANK)
q8 = """
WITH product_revenue AS (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_name
)
SELECT
    category,
    product_name,
    ROUND(total_revenue, 2) AS total_revenue,
    DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
FROM product_revenue
ORDER BY category, rank_in_category;
"""
pd.read_sql_query(q8, conn)

,category,product_name,total_revenue,rank_in_category
0,Books,Because Non-Fiction,35185.08,1
1,Books,Hope Non-Fiction,34912.06,2
2,Books,Interest Fiction,33690.57,3
3,Books,Color Academic,32443.02,4
4,Books,Deal Non-Fiction,32026.66,5
...,...,...,...,...
145,Home,Choose Kitchen,16984.71,28
146,Home,Friend Furniture,16759.67,29
147,Home,General Furniture,15883.37,30
148,Home,Yet Decor,12214.52,31


In [31]:
#Query 9 — Days between consecutive orders (LAG), flag At Risk
q9 = """
WITH customer_orders AS (
    SELECT
        customer_id,
        order_date,
        LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
    FROM orders
    WHERE customer_id IS NOT NULL
),
gaps AS (
    SELECT
        customer_id,
        order_date,
        previous_order_date,
        CASE WHEN previous_order_date IS NOT NULL
             THEN julianday(order_date) - julianday(previous_order_date)
        END AS days_gap
    FROM customer_orders
),
avg_gap AS (
    SELECT customer_id, AVG(days_gap) AS avg_days_gap
    FROM gaps WHERE days_gap IS NOT NULL
    GROUP BY customer_id
)
SELECT
    g.customer_id, g.order_date, g.previous_order_date,
    ROUND(g.days_gap, 1) AS days_gap,
    CASE WHEN a.avg_days_gap > 30 THEN 'At Risk' ELSE 'Active' END AS risk_flag
FROM gaps g
LEFT JOIN avg_gap a ON a.customer_id = g.customer_id
ORDER BY g.customer_id, g.order_date;
"""
pd.read_sql_query(q9, conn)

,customer_id,order_date,previous_order_date,days_gap,risk_flag
0,1,2023-08-16 22:19:03,NaN,NaN,At Risk
1,1,2024-01-21 18:40:39,2023-08-16 22:19:03,157.8,At Risk
2,2,2023-03-16 19:04:02,NaN,NaN,At Risk
3,2,2023-11-28 12:55:05,2023-03-16 19:04:02,256.7,At Risk
4,2,2024-04-27 00:00:00,2023-11-28 12:55:05,150.5,At Risk
...,...,...,...,...,...
1888,599,2024-11-08 01:35:45,2024-10-04 00:42:50,35.0,At Risk
1889,599,2025-02-13 21:37:47,2024-11-08 01:35:45,97.8,At Risk
1890,599,2025-11-13 20:36:39,2025-02-13 21:37:47,273.0,At Risk
1891,600,2023-03-12 16:59:28,NaN,NaN,At Risk


In [32]:
#Query 10 — Multi-level CTE: monthly revenue → category → count per month
q10 = """
WITH monthly_customer_revenue AS (
    SELECT
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS order_month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS monthly_revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id, order_month
),
categorized AS (
    SELECT
        customer_id, order_month, monthly_revenue,
        CASE
            WHEN monthly_revenue > 10000 THEN 'High'
            WHEN monthly_revenue >= 5000 THEN 'Medium'
            ELSE 'Low'
        END AS revenue_category
    FROM monthly_customer_revenue
)
SELECT order_month, revenue_category, COUNT(DISTINCT customer_id) AS customer_count
FROM categorized
GROUP BY order_month, revenue_category
ORDER BY order_month, revenue_category;
"""
pd.read_sql_query(q10, conn)

,order_month,revenue_category,customer_count
0,2023-01,Low,46
1,2023-01,Medium,3
2,2023-02,Low,40
3,2023-02,Medium,2
4,2023-03,High,1
...,...,...,...
70,2025-10,Medium,5
71,2025-11,Low,41
72,2025-11,Medium,5
73,2025-12,Low,45


In [33]:
#Query 11 — NTILE(4) quartile segmentation
q11 = """
WITH customer_value AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_value
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
)
SELECT
    customer_id,
    ROUND(total_value, 2) AS total_value,
    NTILE(4) OVER (ORDER BY total_value DESC) AS quartile,
    CASE NTILE(4) OVER (ORDER BY total_value DESC)
        WHEN 1 THEN 'Platinum' WHEN 2 THEN 'Gold'
        WHEN 3 THEN 'Silver' WHEN 4 THEN 'Bronze'
    END AS quartile_label
FROM customer_value
ORDER BY quartile, total_value DESC;
"""
pd.read_sql_query(q11, conn)

,customer_id,total_value,quartile,quartile_label
0,442,28573.66,1,Platinum
1,255,25552.75,1,Platinum
2,41,24108.70,1,Platinum
3,387,20804.82,1,Platinum
4,537,19156.47,1,Platinum
...,...,...,...,...
568,460,19.98,4,Bronze
569,22,3.01,4,Bronze
570,283,-216.38,4,Bronze
571,264,-365.85,4,Bronze


In [34]:
# Query 12 — Year-over-year monthly revenue comparison
q12 = """
WITH monthly_revenue AS (
    SELECT
        CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,
        CAST(strftime('%m', o.order_date) AS INTEGER) AS month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY year, month
)
SELECT
    curr.year, curr.month,
    ROUND(curr.revenue, 2) AS revenue,
    ROUND(prev.revenue, 2) AS prev_year_revenue,
    CASE
        WHEN prev.revenue IS NULL OR prev.revenue = 0 THEN NULL
        ELSE ROUND((curr.revenue - prev.revenue) / prev.revenue * 100, 2)
    END AS yoy_growth_percent
FROM monthly_revenue curr
LEFT JOIN monthly_revenue prev
    ON prev.year = curr.year - 1 AND prev.month = curr.month
ORDER BY curr.year, curr.month;
"""
pd.read_sql_query(q12, conn)

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2023,1,103618.83,NaN,NaN
1,2023,2,95467.90,NaN,NaN
2,2023,3,134031.57,NaN,NaN
3,2023,4,87720.99,NaN,NaN
4,2023,5,100653.46,NaN,NaN
5,2023,6,123869.31,NaN,NaN
6,2023,7,102614.91,NaN,NaN
7,2023,8,74072.67,NaN,NaN
8,2023,9,99294.62,NaN,NaN
9,2023,10,120350.04,NaN,NaN


In [35]:
# Query 13 — First vs most recent purchased category (category_shift)
q13 = """
WITH customer_category_orders AS (
    SELECT
        o.customer_id, p.category, o.order_date,
        ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date ASC)  AS rn_first,
        ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date DESC) AS rn_last
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.customer_id IS NOT NULL
),
first_cat AS (SELECT customer_id, category AS first_category FROM customer_category_orders WHERE rn_first = 1),
last_cat  AS (SELECT customer_id, category AS last_category  FROM customer_category_orders WHERE rn_last = 1)
SELECT
    f.customer_id, f.first_category, l.last_category,
    CASE WHEN f.first_category != l.last_category THEN 'Yes' ELSE 'No' END AS category_shift
FROM first_cat f
JOIN last_cat l ON l.customer_id = f.customer_id
ORDER BY f.customer_id;
"""
pd.read_sql_query(q13, conn)

,customer_id,first_category,last_category,category_shift
0,1,Clothing,Books,Yes
1,2,Clothing,Electronics,Yes
2,3,Electronics,Electronics,No
3,4,Home,Home,No
4,5,Electronics,Books,Yes
...,...,...,...,...
568,596,Books,Books,No
569,597,Books,Books,No
570,598,Clothing,Home,Yes
571,599,Home,Books,Yes


In [36]:
#Query 14 — Cumulative revenue distribution
q14 = """
WITH customer_revenue AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
),
totals AS (SELECT SUM(revenue) AS grand_total FROM customer_revenue)
SELECT
    cr.customer_id,
    ROUND(cr.revenue, 2) AS revenue,
    ROUND(SUM(cr.revenue) OVER (ORDER BY cr.revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS cumulative_revenue,
    ROUND(SUM(cr.revenue) OVER (ORDER BY cr.revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / t.grand_total * 100, 2) AS cumulative_percent
FROM customer_revenue cr
CROSS JOIN totals t
ORDER BY cr.revenue DESC;
"""
pd.read_sql_query(q14, conn)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,442,28573.66,28573.66,0.86
1,255,25552.75,54126.41,1.63
2,41,24108.70,78235.11,2.36
3,387,20804.82,99039.93,2.98
4,537,19156.47,118196.40,3.56
...,...,...,...,...
568,460,19.98,3323195.10,100.04
569,22,3.01,3323198.11,100.04
570,283,-216.38,3322981.73,100.03
571,264,-365.85,3322615.88,100.02


In [37]:
#Query 15 — Cohort analysis (registration month retention)
q15 = """
WITH cohorts AS (
    SELECT customer_id, strftime('%Y-%m', registration_date) AS cohort_month, date(registration_date) AS reg_date
    FROM customers
),
customer_orders AS (
    SELECT
        c.customer_id, c.cohort_month,
        CAST((strftime('%Y', o.order_date) - strftime('%Y', c.reg_date)) * 12
             + (strftime('%m', o.order_date) - strftime('%m', c.reg_date)) AS INTEGER) AS month_offset
    FROM cohorts c
    JOIN orders o ON o.customer_id = c.customer_id
),
cohort_size AS (SELECT cohort_month, COUNT(*) AS cohort_customers FROM cohorts GROUP BY cohort_month),
activity AS (
    SELECT cohort_month, month_offset, COUNT(DISTINCT customer_id) AS active_customers
    FROM customer_orders
    WHERE month_offset BETWEEN 0 AND 3
    GROUP BY cohort_month, month_offset
)
SELECT
    a.cohort_month, a.month_offset, a.active_customers, cs.cohort_customers,
    ROUND(1.0 * a.active_customers / cs.cohort_customers * 100, 2) AS retention_rate_percent
FROM activity a
JOIN cohort_size cs ON cs.cohort_month = a.cohort_month
ORDER BY a.cohort_month, a.month_offset;
"""
pd.read_sql_query(q15, conn)

,cohort_month,month_offset,active_customers,cohort_customers,retention_rate_percent
0,2023-01,1,2,10,20.00
1,2023-01,3,1,10,10.00
2,2023-02,0,1,11,9.09
3,2023-02,1,2,11,18.18
4,2023-02,2,1,11,9.09
...,...,...,...,...,...
102,2025-10,0,2,21,9.52
103,2025-10,2,5,21,23.81
104,2025-11,0,1,22,4.55
105,2025-11,1,1,22,4.55


In [38]:
#Query 16 — Products frequently bought together (self-join)
q16 = """
SELECT
    p1.product_name AS product_a,
    p2.product_name AS product_b,
    COUNT(*) AS times_bought_together
FROM order_items oi1
JOIN order_items oi2
    ON oi1.order_id = oi2.order_id
    AND oi1.product_id < oi2.product_id
JOIN products p1 ON p1.product_id = oi1.product_id
JOIN products p2 ON p2.product_id = oi2.product_id
GROUP BY p1.product_name, p2.product_name
ORDER BY times_bought_together DESC
LIMIT 20;
"""
pd.read_sql_query(q16, conn)

,product_a,product_b,times_bought_together
0,Without Non-Fiction,Office Footwear,5
1,Above Women,Budget Women,4
2,Agency Footwear,Military Fiction,4
3,Ago Decor,Measure Mobile,4
4,Ago Mobile,Suggest Academic,4
5,Amount Fiction,Political Laptop,4
6,Analysis Comic,Would Footwear,4
7,Campaign Bedding,Around Mobile,4
8,Campaign Bedding,Should Fiction,4
9,Card Comic,Glass Decor,4


In [39]:
#Python + SQL Integration
def get_period_summary(conn, start, end):
    cur = conn.cursor()
    cur.execute("""
        SELECT
            COUNT(DISTINCT o.order_id) AS total_orders,
            COALESCE(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 0) AS total_revenue,
            COUNT(DISTINCT o.customer_id) AS unique_customers
        FROM orders o
        LEFT JOIN order_items oi ON oi.order_id = o.order_id
        WHERE date(o.order_date) BETWEEN date(?) AND date(?)
    """, (start, end))
    total_orders, total_revenue, unique_customers = cur.fetchone()

    cur.execute("""
        SELECT p.product_name,
               SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.order_id
        JOIN products p ON p.product_id = oi.product_id
        WHERE date(o.order_date) BETWEEN date(?) AND date(?)
        GROUP BY p.product_name
        ORDER BY revenue DESC
        LIMIT 3
    """, (start, end))
    top_products = cur.fetchall()

    return {
        "total_orders": total_orders or 0,
        "total_revenue": round(total_revenue or 0, 2),
        "unique_customers": unique_customers or 0,
        "top_products": top_products,
    }

def pct_change(curr, prev):
    if prev == 0:
        return None
    return round((curr - prev) / prev * 100, 2)

def previous_period(start, end):
    length = (end - start).days + 1
    prev_end = start - timedelta(days=1)
    prev_start = prev_end - timedelta(days=length - 1)
    return prev_start, prev_end

def generate_report(report_type, start_str, end_str):
    assert report_type in {"daily", "weekly", "monthly"}
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end = datetime.strptime(end_str, "%Y-%m-%d")
    assert end >= start

    current = get_period_summary(conn, start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
    prev_start, prev_end = previous_period(start, end)
    previous = get_period_summary(conn, prev_start.strftime("%Y-%m-%d"), prev_end.strftime("%Y-%m-%d"))

    print("=" * 55)
    print(f"{report_type.upper()} REPORT: {start.date()} to {end.date()}")
    print("=" * 55)
    print(f"Total Orders     : {current['total_orders']}")
    print(f"Total Revenue    : {current['total_revenue']:.2f}")
    print(f"Unique Customers : {current['unique_customers']}")
    print("\nTop 3 Products:")
    for i, (name, revenue) in enumerate(current["top_products"], start=1):
        print(f"  {i}. {name} - {revenue:.2f}")
    print(f"\nComparison with previous period ({prev_start.date()} to {prev_end.date()}):")
    for metric in ("total_orders", "total_revenue", "unique_customers"):
        change = pct_change(current[metric], previous[metric])
        change_str = f"{change:+.2f}%" if change is not None else "N/A"
        print(f"  {metric:17}: {current[metric]} vs {previous[metric]}  ({change_str})")
    print("=" * 55)
    return current, previous

print("Report function ready.")

Report function ready.


In [40]:
# own report — 
report_type = "weekly"   # daily / weekly / monthly
start_date = "2024-03-01"
end_date = "2024-03-07"

_ = generate_report(report_type, start_date, end_date)

WEEKLY REPORT: 2024-03-01 to 2024-03-07
Total Orders     : 14
Total Revenue    : 23587.55
Unique Customers : 10

Top 3 Products:
  1. Doctor Decor - 4065.11
  2. Color Academic - 3104.70
  3. Class Non-Fiction - 2420.88

Comparison with previous period (2024-02-23 to 2024-02-29):
  total_orders     : 14 vs 13  (+7.69%)
  total_revenue    : 23587.55 vs 22306.3  (+5.74%)
  unique_customers : 10 vs 12  (-16.67%)


In [41]:
#Edge Case Handling
def test_order_items_with_nonexistent_order_id():
    """What happens when order_items has an order_id not in orders?
    -> check_referential_integrity() flags it; such rows are excluded
       from the SQLite load (FK constraint) but preserved in the cleaned
       CSV + report, so nothing silently disappears."""
    orders_t = pd.DataFrame({"order_id": [1, 2, 3]})
    items_t = pd.DataFrame({
        "item_id": [1, 2, 3],
        "order_id": [1, 2, 999],
        "product_id": [10, 11, 12],
        "quantity": [1, 2, 1],
    })
    bad = check_referential_integrity(orders_t, items_t)
    assert len(bad) == 1 and bad.iloc[0]["order_id"] == 999
    print("PASS: order_id=999 correctly flagged as orphaned.")


def test_discount_percent_greater_than_100():
    """What happens when discount_percent > 100?
    -> clean_order_items() clips it to 100 and reports the count,
       instead of silently producing negative revenue."""
    df = pd.DataFrame({
        "item_id": [1, 2], "order_id": [1, 1], "product_id": [10, 11],
        "quantity": [1, 1], "unit_price": [100.0, 50.0],
        "discount_percent": [150.0, 40.0],
    })
    cleaned, issues = clean_order_items(df)
    assert issues["discount_percent_out_of_range"] == 1
    assert cleaned.loc[0, "discount_percent"] == 100.0
    print("PASS: discount_percent=150 clipped to 100.")


def test_quantity_is_zero():
    """What happens when quantity is 0?
    -> Contributes 0 revenue, is neither a purchase nor a return, and is
       counted separately in the report rather than dropped."""
    df = pd.DataFrame({
        "item_id": [1], "order_id": [1], "product_id": [10],
        "quantity": [0], "unit_price": [100.0], "discount_percent": [10.0],
    })
    cleaned, issues = clean_order_items(df)
    assert issues["zero_quantity_rows"] == 1
    revenue = cleaned.loc[0, "quantity"] * cleaned.loc[0, "unit_price"]
    assert revenue == 0
    print("PASS: quantity=0 contributes zero revenue, reported not dropped.")


def test_order_date_in_future():
    """What happens when order_date is in the future?
    -> clean_orders() doesn't reject it outright (could be a legitimate
       pre-order), but it parses correctly and is easy to detect
       downstream via a simple filter."""
    future_date = (datetime.now() + timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")
    df = pd.DataFrame({
        "order_id": [1], "customer_id": ["5"], "order_date": [future_date],
        "status": ["PLACED"], "region_code": ["NORTH"],
    })
    cleaned, _ = clean_orders(df)
    future_mask = cleaned["order_date"] > pd.Timestamp.now()
    assert future_mask.sum() == 1
    print(f"PASS: future order_date ({future_date}) parses and is detectable.")


test_order_items_with_nonexistent_order_id()
test_discount_percent_greater_than_100()
test_quantity_is_zero()
test_order_date_in_future()
print("\nAll edge case tests passed.")



PASS: order_id=999 correctly flagged as orphaned.
PASS: discount_percent=150 clipped to 100.
PASS: quantity=0 contributes zero revenue, reported not dropped.
PASS: future order_date (2026-09-01 18:32:48) parses and is detectable.

All edge case tests passed.


In [42]:
#Cleanup
conn.close()
print("Connection closed. Done!")

Connection closed. Done!
